# Model Exploration: Regime-Switching Stochastic Volatility

This notebook provides an interactive introduction to the regime-switching stochastic volatility framework for exotic options pricing.

## Contents
1. Model Setup
2. Asset Price Simulation
3. Regime Dynamics
4. Option Pricing
5. Visualization

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add project to path
sys.path.insert(0, str(Path.cwd().parent))

from src.models.regime_switching import RegimeSwitchingModel, RegimeParameters
from src.models.asset_dynamics import AssetSimulator
from src.pricing.exotic_options import BarrierOption, VanillaOption
from src.pricing.monte_carlo import MonteCarloEngine
from src.analytics.visualization import ResultsVisualizer
from src.utils.data_utils import load_config

# Plotting setup
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 8)
%matplotlib inline

print("✓ Imports successful")

## 1. Model Setup

We'll create a 3-regime model representing:
- **Regime 0**: Low volatility (calm market)
- **Regime 1**: Medium volatility (normal market)
- **Regime 2**: High volatility (turbulent market)

In [ ]:
# Define regime parameters
regime_params = [
    RegimeParameters(0, "Low Volatility", drift=0.08, volatility=0.15),
    RegimeParameters(1, "Medium Volatility", drift=0.10, volatility=0.25),
    RegimeParameters(2, "High Volatility", drift=0.12, volatility=0.45)
]

# Transition probability matrix
# Rows sum to 1, Q[i,j] = P(next regime = j | current regime = i)
transition_matrix = np.array([
    [0.85, 0.12, 0.03],  # From low vol
    [0.10, 0.80, 0.10],  # From medium vol
    [0.05, 0.25, 0.70]   # From high vol
])

# Create regime-switching model
regime_model = RegimeSwitchingModel(
    regime_params,
    transition_matrix,
    dt=1.0/252  # Daily time step
)

print(regime_model.summary())

## 2. Asset Price Simulation

Create an asset simulator and generate sample paths.

In [ ]:
# Create asset simulator
simulator = AssetSimulator(
    regime_model,
    spot_price=100.0,
    risk_free_rate=0.03,
    dividend_yield=0.01
)

# Simulate paths
n_paths = 1000
n_steps = 252  # One year, daily steps
T = 1.0

prices, regimes, variances = simulator.simulate_paths(
    n_paths=n_paths,
    n_steps=n_steps,
    T=T,
    initial_regime=1,  # Start in medium volatility
    risk_neutral=True,
    seed=42
)

print(f"✓ Simulated {n_paths} price paths")
print(f"  Price range: [{prices.min():.2f}, {prices.max():.2f}]")
print(f"  Final mean price: {prices[:, -1].mean():.2f}")

## 3. Visualize Price Paths and Regimes

In [ ]:
# Plot sample paths
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
time_grid = np.linspace(0, T, n_steps + 1)

# Plot 20 random price paths
sample_indices = np.random.choice(n_paths, 20, replace=False)
for idx in sample_indices:
    axes[0].plot(time_grid, prices[idx, :], alpha=0.5, linewidth=1)

axes[0].axhline(100, color='black', linestyle='--', label='Initial Price')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Asset Price')
axes[0].set_title('Sample Asset Price Paths')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot regime evolution for one path
path_idx = 0
regime_colors = ['green', 'orange', 'red']
axes[1].plot(time_grid, regimes[path_idx, :], drawstyle='steps-post', linewidth=2)
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Regime')
axes[1].set_title('Regime Evolution (Sample Path)')
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['Low Vol', 'Medium Vol', 'High Vol'])
axes[1].grid(True, alpha=0.3)

# Plot terminal price distribution
axes[2].hist(prices[:, -1], bins=50, alpha=0.7, edgecolor='black', density=True)
axes[2].axvline(prices[:, -1].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {prices[:, -1].mean():.2f}')
axes[2].axvline(100, color='black', linestyle='--', 
                linewidth=2, label='Initial: 100')
axes[2].set_xlabel('Terminal Price')
axes[2].set_ylabel('Density')
axes[2].set_title('Terminal Price Distribution')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Regime Statistics

In [ ]:
# Compute regime statistics
regime_stats = simulator.get_regime_statistics(regimes)

print("Regime Statistics:")
print("=" * 50)
print("\nTime spent in each regime:")
for i, time in enumerate(regime_stats['time_in_regime']):
    print(f"  Regime {i}: {time*100:.2f}%")

print(f"\nStationary distribution (theoretical):")
for i, prob in enumerate(regime_stats['stationary_distribution']):
    print(f"  Regime {i}: {prob*100:.2f}%")

print(f"\nAverage transitions per path: {regime_stats['avg_transitions_per_path']:.2f}")
print(f"Average time between transitions: {regime_stats['avg_time_between_transitions']:.2f} steps")

## 5. Option Pricing

Price a barrier option under the regime-switching model.

In [ ]:
# Create Monte Carlo pricing engine
mc_engine = MonteCarloEngine(simulator, n_simulations=50000, seed=42)

# Define up-and-out barrier call
barrier_option = BarrierOption(
    strike=100,
    barrier=120,
    maturity=1.0,
    option_type='call',
    barrier_type='up-and-out',
    rebate=0.0
)

print(f"Option: {barrier_option}")
print("\nPricing...")

# Price the option
result = mc_engine.price_option(barrier_option, antithetic=True)

print("\nResults:")
print("=" * 50)
print(f"Option Price: ${result['price']:.4f}")
print(f"Std Error:    ${result['std_error']:.4f}")
print(f"95% CI:       [${result['ci_95_lower']:.4f}, ${result['ci_95_upper']:.4f}]")
print(f"Mean Payoff:  ${result['mean_payoff']:.4f}")

## 6. Compare with Vanilla Option

In [ ]:
# Price vanilla call for comparison
vanilla_option = VanillaOption(strike=100, maturity=1.0, option_type='call')

vanilla_result = mc_engine.price_option(vanilla_option, antithetic=True)

# Black-Scholes comparison
bs_comparison = mc_engine.compare_with_black_scholes(vanilla_option)

print("Vanilla Call Option:")
print("=" * 50)
print(f"Regime-Switching Price: ${vanilla_result['price']:.4f}")
print(f"Black-Scholes Price:    ${bs_comparison['black_scholes_price']:.4f}")
print(f"Difference:             ${bs_comparison['difference']:.4f}")
print(f"Relative Difference:    {bs_comparison['relative_difference']*100:.2f}%")

print("\nBarrier vs Vanilla:")
print("=" * 50)
print(f"Barrier Option: ${result['price']:.4f}")
print(f"Vanilla Option: ${vanilla_result['price']:.4f}")
print(f"Barrier Discount: {(1 - result['price']/vanilla_result['price'])*100:.2f}%")

## 7. Martingale Validation

In [ ]:
# Validate that discounted prices are martingales
from src.analytics.validation import MartingaleValidator

validator = MartingaleValidator(simulator)
martingale_test = validator.test_martingale_property(n_paths=10000, T=1.0)

print("Martingale Property Test:")
print("=" * 50)
print(f"S_0:                    ${martingale_test['S0']:.4f}")
print(f"E[S_T * exp(-rT)]:      ${martingale_test['mean_discounted_ST']:.4f}")
print(f"Absolute Error:         ${martingale_test['absolute_error']:.4f}")
print(f"Relative Error:         {martingale_test['relative_error']*100:.2f}%")
print(f"P-value:                {martingale_test['p_value']:.4f}")
print(f"Test Result:            {'PASSED ✓' if martingale_test['test_passed'] else 'FAILED ✗'}")

## Summary

This notebook demonstrated:
1. ✓ Setting up a regime-switching model
2. ✓ Simulating asset price paths
3. ✓ Analyzing regime dynamics
4. ✓ Pricing exotic options (barrier)
5. ✓ Comparing with Black-Scholes
6. ✓ Validating the risk-neutral measure

**Next steps**: Explore hedging strategies in notebook `03_hedging_strategies.ipynb`